# HELM: Holistic Evaluation of Language Models

Aggregate benchmark scores hide critical information: a model might excel on accuracy while being poorly calibrated, slower, or more expensive. HELM (Liang et al. 2022) proposed a multi-metric framework that evaluates models across scenarios and metrics simultaneously.

## Why Single-Metric Evaluation Fails

A model with 90% accuracy but 60% calibration error can be dangerous in high-stakes applications — it is confidently wrong. A model with slightly lower accuracy but better calibration, faster inference, and lower cost may be the correct production choice.

Pre-HELM, the community published accuracy on a handful of benchmarks and called it done. HELM's contribution was forcing multi-dimensional evaluation and showing that rankings change significantly depending on which metrics matter for your use case.

## HELM Metrics

HELM evaluated 42 scenarios across 7 metric groups:
- **Accuracy**: task performance (exact match, F1, accuracy)
- **Calibration**: ECE (Expected Calibration Error) — how well predicted confidence matches empirical accuracy
- **Robustness**: performance under perturbations (typos, paraphrases)
- **Fairness**: performance gaps across demographic groups
- **Bias**: frequency of stereotypical associations in model outputs
- **Toxicity**: rate of harmful content generation
- **Efficiency**: inference time and cost per example

In [ ]:
from dataclasses import dataclass, field
import math

@dataclass
class HELMReport:
    model: str
    accuracy: float
    ece: float
    robustness: float
    fairness_gap: float
    toxicity_rate: float
    latency_ms: float

    def composite(self, weights: dict = None) -> float:
        if weights is None:
            weights = {'accuracy': 0.4, 'calibration': 0.2, 'robustness': 0.2, 'safety': 0.2}
        calib_score = max(0, 1 - self.ece * 5)  # ECE 0.0 -> 1.0, ECE 0.2 -> 0.0
        safety_score = max(0, 1 - self.toxicity_rate * 20)
        return (weights['accuracy'] * self.accuracy +
                weights['calibration'] * calib_score +
                weights['robustness'] * self.robustness +
                weights['safety'] * safety_score)

models = [
    HELMReport('ModelA', accuracy=0.87, ece=0.05, robustness=0.83, fairness_gap=0.04, toxicity_rate=0.01, latency_ms=450),
    HELMReport('ModelB', accuracy=0.91, ece=0.18, robustness=0.71, fairness_gap=0.09, toxicity_rate=0.03, latency_ms=380),
    HELMReport('ModelC', accuracy=0.78, ece=0.03, robustness=0.88, fairness_gap=0.02, toxicity_rate=0.005, latency_ms=210),
]

print(f'{'Model':<10} {'Accuracy':>9} {'ECE':>6} {'Robust':>8} {'Tox':>6} {'Composite':>11}')
print('-' * 55)
for m in sorted(models, key=lambda x: -x.composite()):
    print(f'{m.model:<10} {m.accuracy:>9.2%} {m.ece:>6.3f} {m.robustness:>8.2%} {m.toxicity_rate:>6.3f} {m.composite():>11.3f}')

## Expected Calibration Error

Calibration measures whether a model's confidence tracks its actual accuracy. A well-calibrated model that says it is 80% confident should be correct about 80% of the time.

ECE bins predictions by confidence, then measures the weighted average gap between mean confidence and accuracy within each bin.

In [ ]:
def compute_ece(predictions: list, n_bins: int = 10) -> float:
    bins = [[] for _ in range(n_bins)]
    for confidence, correct in predictions:
        bin_idx = min(int(confidence * n_bins), n_bins - 1)
        bins[bin_idx].append((confidence, correct))
    ece = 0.0
    n = len(predictions)
    for b in bins:
        if not b:
            continue
        avg_conf = sum(c for c, _ in b) / len(b)
        avg_acc = sum(x for _, x in b) / len(b)
        ece += (len(b) / n) * abs(avg_conf - avg_acc)
    return ece

# Well-calibrated model
import random
rng = random.Random(42)
well_calibrated = []
for _ in range(500):
    conf = rng.uniform(0.5, 1.0)
    correct = 1 if rng.random() < conf else 0
    well_calibrated.append((conf, correct))

# Overconfident model
overconfident = [(min(1.0, c + 0.2), x) for c, x in well_calibrated]

print(f'Well-calibrated ECE: {compute_ece(well_calibrated):.4f}')
print(f'Overconfident ECE:   {compute_ece(overconfident):.4f}')
print('Lower ECE = better calibration')

## The Multi-Metric Insight

HELM showed that model rankings are not stable across metrics. The model with the highest accuracy often has poor calibration. The most robust model is often slower. The safest model sacrifices helpfulness.

This matters for production: the right model depends on your application. A customer service bot and a medical decision support tool should optimize different HELM metric profiles even if they use the same underlying architecture.

The practical takeaway: define your evaluation criteria before you train or select a model. Post-hoc metric selection creates exactly the kind of benchmarking gaming HELM was designed to prevent.